# 02 - TOKENISATION DES CORPUS

Etape **Tokenisation**. L'entrée est ce que produit le notebook 01 (sous `clean/`). La sortie est une liste de tokens par livre, déposée sous `tokens/`. La partie POS / lemmes vient au notebook suivant.

**Choix techniques** : on utilise spaCy (`spacy.blank("fr")`) plutot qu'un `split` simple, parce que le français a des contractions sur apostrophe (`l'echo`, `j'ai`, `qu'il`) et des traits d'union pronominaux (`dit-elle`, `allez-vous-en`). spaCy les gère correctement de base. `spacy.blank` ne charge que le tokenizer.

La table de normalisation typographique (apostrophes courbes -> droites, tirets cadratins -> simples, guillemets -> ASCII) est définie une fois pour toutes dans `pipeline.tokenization` et réutilisée à l'annotation. Sans elle, `l'echo` (apostrophe ASCII) et `l'echo` (apostrophe typographique) seraient deux tokens distincts dans les fréquences.


## Setup

In [1]:
import sys
from pathlib import Path

# Se déplacer dans le dossier du projet (NE LANCER Q'UNE FOIS)
%cd ../
%ls

/home/gau/projets/nlp/projet_nlp_app
README.md  figures/  notebooks/      pipeline/         scripts/
app/       models/   pg_catalog.csv  requirements.txt


In [2]:
from collections import Counter

from pipeline import storage
from pipeline.config import PREFIXES
from pipeline.tokenization import tokeniser, normaliser, _get_tokenizer

nlp = _get_tokenizer()
print(f"Tokenizer charge : lang={nlp.lang}, pipeline={nlp.pipe_names}")

Tokenizer charge : lang=fr, pipeline=[]


## Vérification rapide sur des cas français délicats

On s'assure que les apostrophes et les pronoms en trait d'union sont découpés proprement avant de lancer sur les vrais livres.


In [4]:
exemples = [
    "L'echo de la foret resonnait.",
    "J'ai aujourd'hui rencontre M. Dupont.",
    "Qu'est-ce que c'est ?",
    "Allez-vous-en !",
    "Il faut qu'il vienne, dit-elle.",
]

exemples2 = [
    """--Pauvre père!» dit Monte-Cristo.

    Le comte continua:

    --«Je lui rends l'espoir, je lui rends la vie, monsieur le comte, en lui
    annonçant que ce fils, que depuis quinze ans il cherche vainement, vous
    pouvez le lui faire retrouver.»

    Le Lucquois regarda Monte-Cristo avec une indéfinissable expression
    d'inquiétude.

    «Je le puis», répondit Monte-Cristo.

    Le major se redressa."""
]

for ex in exemples:
    print(ex)
    print("  ->", tokeniser(ex))
    print()

for ex2 in exemples2:
    print(ex2)
    print("  ->", tokeniser(ex2))
    print()

L'echo de la foret resonnait.
  -> ["L'", 'echo', 'de', 'la', 'foret', 'resonnait', '.']

J'ai aujourd'hui rencontre M. Dupont.
  -> ["J'", 'ai', "aujourd'hui", 'rencontre', 'M.', 'Dupont', '.']

Qu'est-ce que c'est ?
  -> ["Qu'", 'est', '-ce', 'que', "c'", 'est', '?']

Allez-vous-en !
  -> ['Allez', '-vous', '-en', '!']

Il faut qu'il vienne, dit-elle.
  -> ['Il', 'faut', "qu'", 'il', 'vienne', ',', 'dit', '-elle', '.']

--Pauvre père!» dit Monte-Cristo.

    Le comte continua:

    --«Je lui rends l'espoir, je lui rends la vie, monsieur le comte, en lui
    annonçant que ce fils, que depuis quinze ans il cherche vainement, vous
    pouvez le lui faire retrouver.»

    Le Lucquois regarda Monte-Cristo avec une indéfinissable expression
    d'inquiétude.

    «Je le puis», répondit Monte-Cristo.

    Le major se redressa.
  -> ['--Pauvre', 'père', '!', '"', 'dit', 'Monte', '-', 'Cristo', '.', 'Le', 'comte', 'continua', ':', '--"Je', 'lui', 'rends', "l'", 'espoir', ',', 'je', 'lui', 're

## Lister les livres deja nettoyés

In [4]:
livres_propres = storage.list_objects(PREFIXES["clean"], suffix=".txt")
print(f"{len(livres_propres)} livres nettoyes a tokeniser")
for o in livres_propres[:5]:
    print(" -", o["Key"])

26 livres nettoyes a tokeniser
 - clean/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt
 - clean/adventure/raspe_rudolf_erich/pg50398_aventures_de_baron_de_munchausen.txt
 - clean/biography/fusil_louise/pg26720_souvenirs_dune_actrice_23.txt
 - clean/biography/marmont_auguste_frederic_louis_viesse_de_duc_de_raguse/pg30013_memoires_du_marechal_marmont_duc_de_raguse_39.txt
 - clean/biography/rouquette_louis_frederic/pg70801_lepopee_blanche.txt


## Test sur un petit extrait (5 000 caracteres)

On commence par tokeniser seulement le debut d'un livre pour vérifier que tout se comporte bien sans attendre le traitement complet.


In [5]:
cle_test = livres_propres[0]["Key"]
texte = storage.get_text(cle_test)

extrait = texte[:5000]
tokens = tokeniser(extrait)

print(f"Livre  : {cle_test}")
print(f"Extrait: {len(extrait)} caracteres")
print(f"Tokens : {len(tokens)}")
print(f"Vocab  : {len(set(tokens))} tokens uniques\n")

print("--- 30 premiers tokens ---")
print(tokens[:30])
print()
print("--- Top 10 frequences ---")
for tok, n in Counter(tokens).most_common(10):
    print(f"  {n:5d}  {tok!r}")

Livre  : clean/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt
Extrait: 5000 caracteres
Tokens : 1015
Vocab  : 457 tokens uniques

--- 30 premiers tokens ---
['LE', 'COMTE', 'DE', 'MONTE', '-', 'CRISTO', 'Alexandre', 'Dumas', 'Tome', 'II', '(', '1845', '-', '1846', ')', 'Table', 'des', 'matières', 'XXXII', 'Réveil', '.', 'XXXIII', 'Bandits', 'romains', '.', 'XXXIV', 'Apparition', '.', 'XXXV', 'La']

--- Top 10 frequences ---
     58  ','
     45  '.'
     37  'de'
     25  'et'
     23  'le'
     20  'la'
     19  'son'
     18  'il'
     17  'à'
     15  "d'"


## Test sur le livre complet

In [6]:
tokens = tokeniser(texte)

print(f"Livre      : {cle_test}")
print(f"Caracteres : {len(texte):>8}")
print(f"Tokens     : {len(tokens):>8}")
print(f"Vocabulaire: {len(set(tokens)):>8}")
print(f"Ratio chars/token : {len(texte) / len(tokens):.2f}")
print()
print("--- Top 15 frequences (livre entier) ---")
for tok, n in Counter(tokens).most_common(15):
    print(f"  {n:6d}  {tok!r}")

Livre      : clean/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt
Caracteres :   723241
Tokens     :   149856
Vocabulaire:    11843
Ratio chars/token : 4.83

--- Top 15 frequences (livre entier) ---
   11218  ','
    4842  'de'
    4717  '.'
    2838  'le'
    2694  'la'
    2672  'et'
    2665  'à'
    1990  'que'
    1812  'un'
    1807  'vous'
    1788  "l'"
    1556  'il'
    1524  '"'
    1401  'en'
    1384  ';'


## Tokeniser tous les livres et uploader sous `tokens/`

On miroite l'arborescence : `clean/genre/auteur/livre.txt` devient `tokens/genre/auteur/livre.json`. Un fichier JSON par livre = une liste de chaines, dans l'ordre du texte.


In [7]:
for obj in livres_propres:
    cle = obj["Key"]
    texte = storage.get_text(cle)
    tokens = tokeniser(texte)

    sous_cle = cle[len(PREFIXES["clean"]):]
    if sous_cle.endswith(".txt"):
        sous_cle = sous_cle[:-4] + ".json"
    nouvelle_cle = PREFIXES["tokens"] + sous_cle

    storage.put_json(
        nouvelle_cle, tokens,
        metadata={
            "source":       "tokens",
            "original_key": cle,
            "n_tokens":     str(len(tokens)),
        },
    )

    print(f"{cle}: {len(texte):>7} chars -> {len(tokens):>7} tokens "
          f"({len(set(tokens)):>5} uniques)")

clean/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt:  723241 chars ->  149856 tokens (11843 uniques)
clean/adventure/raspe_rudolf_erich/pg50398_aventures_de_baron_de_munchausen.txt:  160892 chars ->   32128 tokens ( 5470 uniques)
clean/biography/fusil_louise/pg26720_souvenirs_dune_actrice_23.txt:  328687 chars ->   65807 tokens ( 8360 uniques)
clean/biography/marmont_auguste_frederic_louis_viesse_de_duc_de_raguse/pg30013_memoires_du_marechal_marmont_duc_de_raguse_39.txt:  682460 chars ->  133225 tokens (10099 uniques)
clean/biography/rouquette_louis_frederic/pg70801_lepopee_blanche.txt:  256902 chars ->   52877 tokens ( 8147 uniques)
clean/biography/savary_anne_jean_marie_rene_duc_de_rovigo/pg20895_memoires_du_duc_de_rovigo_pour_servir_a_lhistoire_de_lempereur_napoleon_tome_2.txt:  589960 chars ->  114808 tokens ( 9541 uniques)
clean/biography/stendhal/pg30977_la_vie_de_rossini_tome_i.txt:  401019 chars ->   80272 tokens ( 9194 uniques)
clean/historical_fiction